In [62]:
import numpy as np
import keras
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, SimpleRNN, Dense, Input ,TextVectorization


In [63]:
# 1. Create a dummy corpus (Text + Labels) # NOTE:  Wrap within numpy array to avoid model fitting issues
sentences = np.array([
    "i love this movie",
    "this film was fantastic",
    "absolutely great acting",
    "i hate this movie",
    "this film was terrible",
    "boring and awful acting",
    "This movie was beautiful",
    "A really underrated movie"
],dtype=object)
#   1 = Positive sentiment, 0 = Negative sentiment ~ For sentiment Analysis
labels = np.array([1, 1, 1, 0, 0, 0, 1, 1])


In [64]:
# 2. Configure the modern TextVectorization layer
max_vocab = 100         # Maximum unique words allowed ~ Limit the words when u want to
max_length = 4          # Forces all sequences to pad/truncate to exactly 4 words , meaning each vector will be exactly of len = 4

# This layer is used for conversion of the sentences to vectors with the passed params
vectorize_layer = TextVectorization(
    max_tokens=max_vocab,
    output_mode='int',                # Output integer sequences
    output_sequence_length=max_length # Automatically handles padding/truncating
)

In [65]:
# Now pass the sentences to the vecotorizer so that it can fit/adapt on them
vectorize_layer.adapt(sentences)

# Why adapt first ?
#     We adapt the vectorizer first because the model needs to build its vocabulary index before it can process any text.
#     Without this step, the layer has no dictionary to look up words and map them to numbers.

In [66]:
# Build the complete model :
model = Sequential([
    # Accept raw string tensors directly as input
    Input(shape=(1,), dtype="string"),

    # 1st layer: Turn string text into sequence of integers
    vectorize_layer, # We have this already adapted with our params (Creation of the dictionary for each word and it's vector)

    # 2nd layer: Turn integers into multi-dimensional vectors for the RNN
    Embedding(input_dim=max_vocab, output_dim=8), # Without this we would have had to OneHotEncode or use another method manually

    # 3rd layer: Process sequences through the RNN
    SimpleRNN(units=16, activation='tanh'), # A simple RNN class with the activation function passes as tanh(general).

    # 4th layer: Binary classifier output
    Dense(units=1, activation='sigmoid') # The final output layer to predict the next word
])

In [67]:
# 4. Compile and Train
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(sentences, labels, epochs=5, batch_size=2)

Epoch 1/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.3750 - loss: 0.7029  
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5000 - loss: 0.6856
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6250 - loss: 0.6789
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7500 - loss: 0.6647    
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8750 - loss: 0.6535


In [68]:
# Let's evaluate our trained model :

test_sentences = np.array([
    "highly recommended film",
    "waste of time and boring"
], dtype=object)

test_labels = np.array([1.0, 0.0], dtype="float32")

# 2. Run evaluation
loss, accuracy = model.evaluate(test_sentences, test_labels)
print(f"\nTest Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.5000 - loss: 0.6910

Test Loss: 0.6910
Test Accuracy: 50.00%
